# HackMate – Smart Hackathon Management MVP
A simple prototype for team registration, project submissions, judging, and leaderboard generation.

**Scoring:** Innovation (20), Working Demo (25), Technical Skills (25), Usefulness (15), Presentation (15). Total = 100.

In [ ]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets

teams = {}
scores = []

def register_team(team_id, team_name, leader, email):
    if team_id in teams:
        return 'This Team ID is already registered.'
    teams[team_id] = {'Team ID': team_id, 'Team Name': team_name, 'Leader': leader,
                      'Email': email, 'Project Link': '', 'Slides Link': ''}
    return f'Success! {team_name} registered with Team ID {team_id}.'

def submit_project(team_id, project_link, slides_link=''):
    if team_id not in teams:
        return 'Team ID not found. Please register first.'
    teams[team_id]['Project Link'] = project_link
    teams[team_id]['Slides Link'] = slides_link
    return 'Project details submitted successfully.'

def add_judging_score(team_id, judge, innovation, demo, technical, usefulness, presentation):
    if team_id not in teams:
        return 'Team ID not found.'
    marks = [innovation, demo, technical, usefulness, presentation]
    limits = [20, 25, 25, 15, 15]
    if any(m < 0 or m > limit for m, limit in zip(marks, limits)):
        return 'Invalid marks. Please check the maximum marks for each criterion.'
    scores.append({'Team ID': team_id, 'Judge': judge, 'Innovation': innovation,
                   'Working Demo': demo, 'Technical Skills': technical,
                   'Usefulness': usefulness, 'Presentation': presentation,
                   'Total': sum(marks)})
    return 'Judging score saved.'

def show_leaderboard():
    if not scores:
        print('No judging scores have been submitted yet.')
        return
    df = pd.DataFrame(scores)
    board = df.groupby('Team ID', as_index=False)['Total'].mean()
    board = board.merge(pd.DataFrame(teams.values())[['Team ID','Team Name']], on='Team ID')
    board = board.rename(columns={'Total':'Average Score'})
    board = board.sort_values('Average Score', ascending=False).reset_index(drop=True)
    board.insert(0, 'Rank', board.index + 1)
    display(board)
    return board

def show_teams():
    if not teams:
        print('No teams registered yet.')
    else:
        display(pd.DataFrame(teams.values()))

print('HackMate functions are ready! Use the forms below or call the functions directly.')

## 1. Team Registration
Enter a unique Team ID for each team. The prototype prevents duplicate IDs.

In [ ]:
team_id = widgets.Text(description='Team ID:')
team_name = widgets.Text(description='Team name:')
leader = widgets.Text(description='Leader:')
email = widgets.Text(description='Email:')
register_btn = widgets.Button(description='Register Team', button_style='success')
register_out = widgets.Output()

def on_register(_):
    with register_out:
        clear_output()
        if not all([team_id.value.strip(), team_name.value.strip(), leader.value.strip(), email.value.strip()]):
            print('Please fill in all fields.')
        else:
            print(register_team(team_id.value.strip(), team_name.value.strip(), leader.value.strip(), email.value.strip()))
register_btn.on_click(on_register)
display(widgets.VBox([team_id, team_name, leader, email, register_btn, register_out]))

## 2. Project Submission
Teams can submit their project repository and presentation links using their Team ID.

In [ ]:
sub_team = widgets.Text(description='Team ID:')
project_link = widgets.Text(description='Project URL:', layout=widgets.Layout(width='95%'))
slides_link = widgets.Text(description='Slides URL:', layout=widgets.Layout(width='95%'))
submit_btn = widgets.Button(description='Submit Project', button_style='info')
submit_out = widgets.Output()

def on_submit(_):
    with submit_out:
        clear_output()
        print(submit_project(sub_team.value.strip(), project_link.value.strip(), slides_link.value.strip()))
submit_btn.on_click(on_submit)
display(widgets.VBox([sub_team, project_link, slides_link, submit_btn, submit_out]))

## 3. Judge Scoring
Each judge scores the same five criteria. The prototype validates the maximum marks.

In [ ]:
judge_team = widgets.Text(description='Team ID:')
judge_name = widgets.Text(description='Judge name:')
innovation = widgets.BoundedIntText(value=15, min=0, max=20, description='Innovation /20')
demo = widgets.BoundedIntText(value=20, min=0, max=25, description='Demo /25')
technical = widgets.BoundedIntText(value=20, min=0, max=25, description='Technical /25')
usefulness = widgets.BoundedIntText(value=10, min=0, max=15, description='Usefulness /15')
presentation = widgets.BoundedIntText(value=10, min=0, max=15, description='Presentation /15')
score_btn = widgets.Button(description='Save Score', button_style='success')
score_out = widgets.Output()

def on_score(_):
    with score_out:
        clear_output()
        print(add_judging_score(judge_team.value.strip(), judge_name.value.strip(), innovation.value,
                                demo.value, technical.value, usefulness.value, presentation.value))
score_btn.on_click(on_score)
display(widgets.VBox([judge_team, judge_name, innovation, demo, technical, usefulness, presentation, score_btn, score_out]))

## 4. View Teams and Live Leaderboard
Run the cell below whenever you want to refresh the team list and rankings.

In [ ]:
print('REGISTERED TEAMS')
show_teams()
print('\nLEADERBOARD')
leaderboard = show_leaderboard()

## 5. Export Backup Files
Download CSV backups of registrations and judge scores. For a real event, also keep a separate shared backup and restrict access to personal data.

In [ ]:
if teams:
    pd.DataFrame(teams.values()).to_csv('hackmate_teams_backup.csv', index=False)
    print('Saved hackmate_teams_backup.csv')
if scores:
    pd.DataFrame(scores).to_csv('hackmate_scores_backup.csv', index=False)
    print('Saved hackmate_scores_backup.csv')
print('Use the Colab file panel to download the CSV backup files.')